# NPC-Sim v4 — Dual-LLM Pipeline Fine-Tuning
**Kaggle GPU: T4 x2 / P100 (16 GB VRAM)**

### Architecture v2.0 — Two-Model Pipeline
| Component | Base Model | Role | Output |
|-----------|-----------|------|--------|
| **A — Reasoner** | Llama 3.2 3B | Turkish CoT reasoning | 3-5 sentence internal monologue |
| **B — Formatter** | Llama 3.2 1B | NL → strict JSON | `{action_id, target_id, dialogue, emotion}` |

### Pipeline
1. Generate v4 datasets (Reasoner corpus + Formatter corpus w/ paraphrase augmentation)
2. **Phase A:** Fine-tune Llama 3.2 3B on CoT reasoning (10k examples)
3. **Phase B:** Fine-tune Llama 3.2 1B on NL→JSON formatting (~12k examples)
4. Evaluate both models independently
5. Export LoRA adapters

## 0. Install Dependencies
Restart kernel & clear outputs after running this cell.

In [ ]:
!pip install -q \
    transformers==4.46.3 \
    datasets==3.1.0 \
    peft==0.13.2 \
    trl==0.12.1 \
    bitsandbytes==0.44.1 \
    accelerate==1.1.1 \
    sentencepiece \
    scipy

!pip install -q -U bitsandbytes==0.43.3
!pip install -q triton==2.3.1

## 1. Generate Dual-LLM Datasets
Runs the v4 generator to produce:
- `train_reasoner.jsonl` — 10k CoT examples for Component A
- `test_reasoner.jsonl` — 2k eval set
- `train_formatter.jsonl` — ~12k (cot → JSON) pairs for Component B (incl. 17.5% paraphrase augmentation)

In [ ]:
import sys, os

SCRIPT_PATH = "/kaggle/input/rpg-dataset-llama-3/Stateful_NPC/generator/npc_sim_generator_v2.py"
DATA_DIR = "/kaggle/working/data"
os.makedirs(DATA_DIR, exist_ok=True)

# Read the script and strip the __main__ block before exec
script = open(SCRIPT_PATH, encoding="utf-8").read()
script = script[:script.find('if __name__')]  # cut off the auto-run block
exec(script)

# -- Reasoner corpus (Component A) --
train_examples = generate_dataset(
    os.path.join(DATA_DIR, "train_reasoner.jsonl"), count=10000, seed=456234
)
test_examples = generate_dataset(
    os.path.join(DATA_DIR, "test_reasoner.jsonl"),  count=2000,  seed=984756
)

# -- Formatter corpus (Component B, derived from train split) --
generate_formatter_dataset(
    os.path.join(DATA_DIR, "train_formatter.jsonl"),
    reasoner_examples=train_examples,
    augment_rate=_PARAPHRASE_RATE,
)

print("\nAll v4 datasets generated!")

## 2. Preview & Validate Datasets

In [ ]:
import json

# --- Reasoner dataset preview ---
with open(os.path.join(DATA_DIR, "train_reasoner.jsonl"), encoding="utf-8") as f:
    first_r = json.loads(f.readline())

print("=" * 60)
print("REASONER TRAINING EXAMPLE (first 600 chars):")
print("=" * 60)
print(first_r["text"][:600])
print("...\n")

# --- Formatter dataset preview ---
with open(os.path.join(DATA_DIR, "train_formatter.jsonl"), encoding="utf-8") as f:
    first_f = json.loads(f.readline())

print("=" * 60)
print("FORMATTER TRAINING EXAMPLE (first 600 chars):")
print("=" * 60)
print(first_f["text"][:600])
print("...\n")

# --- Size report ---
for name in ["train_reasoner.jsonl", "test_reasoner.jsonl", "train_formatter.jsonl"]:
    path = os.path.join(DATA_DIR, name)
    n_lines = sum(1 for _ in open(path, encoding="utf-8"))
    size_mb = os.path.getsize(path) / 1e6
    print(f"  {name:30s}  {n_lines:>6,} examples  {size_mb:.1f} MB")

---
# Phase A: Reasoner (Component A) — Llama 3.2 3B
Fine-tune on Turkish Chain-of-Thought reasoning.

Input: full NPC state JSON → Output: 3-5 sentence Turkish internal monologue.

The model learns **why** an NPC should act, ignoring JSON formatting entirely.

## A.1 Load Reasoner Dataset

In [ ]:
from datasets import load_dataset

ds_reasoner = load_dataset(
    "json",
    data_files={
        "train": os.path.join(DATA_DIR, "train_reasoner.jsonl"),
        "test":  os.path.join(DATA_DIR, "test_reasoner.jsonl"),
    }
)

print(ds_reasoner)
print(f"\nReasonerTrain[0] snippet:")
print(ds_reasoner["train"][0]["text"][:300])

## A.2 Load Llama 3.2 3B + LoRA

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model

REASONER_MODEL_ID = "NousResearch/Hermes-3-Llama-3.2-3B"

tokenizer_a = AutoTokenizer.from_pretrained(REASONER_MODEL_ID)
tokenizer_a.pad_token = tokenizer_a.eos_token
tokenizer_a.padding_side = "right"

model_a = AutoModelForCausalLM.from_pretrained(
    REASONER_MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
)

print(f"Base model loaded: {REASONER_MODEL_ID}")
print(f"Memory: {model_a.get_memory_footprint() / 1e9:.2f} GB")

# --- LoRA ---
lora_config_a = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model_a = get_peft_model(model_a, lora_config_a)
model_a.enable_input_require_grads()
model_a.gradient_checkpointing_enable()

model_a.print_trainable_parameters()

## A.3 Train Reasoner

In [ ]:
from transformers import TrainingArguments
from trl import SFTTrainer

os.environ["TOKENIZERS_PARALLELISM"] = "false"

OUTPUT_DIR_A = "/kaggle/working/reasoner-lora"

training_args_a = TrainingArguments(
    output_dir=OUTPUT_DIR_A,
    num_train_epochs=1,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    fp16=True,
    logging_steps=50,
    eval_strategy="steps",
    eval_steps=250,
    save_strategy="steps",
    save_steps=250,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    report_to="none",
    dataloader_num_workers=2,
    optim="adamw_torch",
    max_grad_norm=0.3,
    group_by_length=True,
    gradient_checkpointing=True,
)

MAX_SEQ_LEN_A = 1024  # NPC state JSON + CoT reasoning

trainer_a = SFTTrainer(
    model=model_a,
    tokenizer=tokenizer_a,
    train_dataset=ds_reasoner["train"],
    eval_dataset=ds_reasoner["test"],
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LEN_A,
    packing=True,
    args=training_args_a,
)

print("[Phase A] SFTTrainer ready. Starting Reasoner training...")
trainer_a.train()

## A.4 Save Reasoner Adapter

In [ ]:
ADAPTER_DIR_A = "/kaggle/working/reasoner-lora-final"

trainer_a.model.save_pretrained(ADAPTER_DIR_A)
tokenizer_a.save_pretrained(ADAPTER_DIR_A)

print(f"Reasoner adapter saved to: {ADAPTER_DIR_A}")
for f in os.listdir(ADAPTER_DIR_A):
    size = os.path.getsize(os.path.join(ADAPTER_DIR_A, f)) / 1e6
    print(f"  {f}: {size:.1f} MB")

## A.5 Evaluate Reasoner
Component A should output Turkish internal monologue (NOT JSON).

Checks:
- Output length is 50-500 chars
- Output does NOT start with `{` (that would mean it's producing JSON instead of reasoning)
- Output contains Turkish language markers

In [ ]:
import json
import torch

BATCH_SIZE = 8
MAX_NEW_TOK = 300
DEVICE = next(model_a.parameters()).device

eval_samples_a = ds_reasoner["test"].select(range(min(100, len(ds_reasoner["test"]))))

ass_tag = "<|start_header_id|>assistant<|end_header_id|>\n\n"

prompts_a = []
for sample in eval_samples_a:
    text = sample["text"]
    p_end = text.find(ass_tag)
    if p_end == -1:
        continue
    prompts_a.append(text[:p_end + len(ass_tag)])

print(f"Evaluating {len(prompts_a)} Reasoner examples...")

good_length = 0
no_json = 0
model_a.eval()

for batch_start in range(0, len(prompts_a), BATCH_SIZE):
    batch = prompts_a[batch_start : batch_start + BATCH_SIZE]

    tokenizer_a.padding_side = "left"
    enc = tokenizer_a(
        batch, return_tensors="pt", padding=True,
        truncation=True, max_length=1024,
    ).to(DEVICE)

    with torch.no_grad():
        out_ids = model_a.generate(
            **enc, max_new_tokens=MAX_NEW_TOK,
            do_sample=False, pad_token_id=tokenizer_a.eos_token_id,
            eos_token_id=tokenizer_a.eos_token_id,
        )

    new_ids = out_ids[:, enc["input_ids"].shape[1]:]
    decoded = tokenizer_a.batch_decode(new_ids, skip_special_tokens=True)

    for gen in decoded:
        gen = gen.split("<|eot_id|>")[0].strip()
        if 50 <= len(gen) <= 600:
            good_length += 1
        if not gen.strip().startswith("{"):
            no_json += 1

    done = min(batch_start + BATCH_SIZE, len(prompts_a))
    print(f"  [{done}/{len(prompts_a)}] good_len={good_length}  no_json={no_json}")

tokenizer_a.padding_side = "right"
n = len(prompts_a)
print(f"\n{'='*50}")
print(f"Reasoner Eval -- {n} examples:")
print(f"  Good length (50-600 chars): {good_length/n*100:.1f}%  (target >=90%)")
print(f"  Non-JSON output (correct):  {no_json/n*100:.1f}%  (target >=95%)")
print(f"{'='*50}")

## A.6 Unload Reasoner (Free VRAM for Phase B)

In [ ]:
import gc

del model_a, trainer_a
gc.collect()
torch.cuda.empty_cache()

print(f"VRAM freed. Allocated: {torch.cuda.memory_allocated()/1e9:.2f} GB")
print("Ready for Phase B.")

---
# Phase B: Formatter (Component B) — Llama 3.2 1B
Fine-tune on Turkish NL → strict JSON extraction.

Input: Component A's Turkish reasoning text → Output: strictly typed JSON response.

The model learns **schema compliance**, not reasoning. It's a "dumb but obedient" translator.

## B.1 Load Formatter Dataset

In [ ]:
from datasets import load_dataset

ds_formatter = load_dataset(
    "json",
    data_files={
        "train": os.path.join(DATA_DIR, "train_formatter.jsonl"),
    }
)

# Use 10% of formatter data as eval split
ds_formatter = ds_formatter["train"].train_test_split(test_size=0.1, seed=42)

print(ds_formatter)
print(f"\nFormatterTrain[0] snippet:")
print(ds_formatter["train"][0]["text"][:400])

## B.2 Load Llama 3.2 1B + LoRA

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model

FORMATTER_MODEL_ID = "meta-llama/Llama-3.2-1B-Instruct"

tokenizer_b = AutoTokenizer.from_pretrained(FORMATTER_MODEL_ID)
tokenizer_b.pad_token = tokenizer_b.eos_token
tokenizer_b.padding_side = "right"

model_b = AutoModelForCausalLM.from_pretrained(
    FORMATTER_MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
)

print(f"Base model loaded: {FORMATTER_MODEL_ID}")
print(f"Memory: {model_b.get_memory_footprint() / 1e9:.2f} GB")

# --- LoRA (smaller rank -- simpler task) ---
lora_config_b = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model_b = get_peft_model(model_b, lora_config_b)
model_b.enable_input_require_grads()
model_b.gradient_checkpointing_enable()

model_b.print_trainable_parameters()

## B.3 Train Formatter

In [ ]:
from transformers import TrainingArguments
from trl import SFTTrainer

OUTPUT_DIR_B = "/kaggle/working/formatter-lora"

training_args_b = TrainingArguments(
    output_dir=OUTPUT_DIR_B,
    num_train_epochs=2,              # formatter task is simpler, 2 epochs
    per_device_train_batch_size=4,   # 1B model = more batch size fits
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=3e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    fp16=True,
    logging_steps=50,
    eval_strategy="steps",
    eval_steps=200,
    save_strategy="steps",
    save_steps=200,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    report_to="none",
    dataloader_num_workers=2,
    optim="adamw_torch",
    max_grad_norm=0.3,
    group_by_length=True,
    gradient_checkpointing=True,
)

# Formatter context is tiny: reasoning text (~100-250 tokens) + JSON output (~100 tokens)
MAX_SEQ_LEN_B = 512

trainer_b = SFTTrainer(
    model=model_b,
    tokenizer=tokenizer_b,
    train_dataset=ds_formatter["train"],
    eval_dataset=ds_formatter["test"],
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LEN_B,
    packing=True,
    args=training_args_b,
)

print("[Phase B] SFTTrainer ready. Starting Formatter training...")
trainer_b.train()

## B.4 Save Formatter Adapter

In [ ]:
ADAPTER_DIR_B = "/kaggle/working/formatter-lora-final"

trainer_b.model.save_pretrained(ADAPTER_DIR_B)
tokenizer_b.save_pretrained(ADAPTER_DIR_B)

print(f"Formatter adapter saved to: {ADAPTER_DIR_B}")
for f in os.listdir(ADAPTER_DIR_B):
    size = os.path.getsize(os.path.join(ADAPTER_DIR_B, f)) / 1e6
    print(f"  {f}: {size:.1f} MB")

## B.5 Evaluate Formatter -- JSON Parse Rate & Action Validity
Component B **must** produce valid JSON with a correct `action_id`.

Targets: JSON parse >= 99%, valid action_id >= 98%

In [ ]:
import json
import torch

VALID_ACTIONS = ["eat","drink","sleep","flee","gather","heal",
                 "attack","socialize","trade","work","pray","walk_to"]

BATCH_SIZE = 16
MAX_NEW_TOK = 200
DEVICE = next(model_b.parameters()).device

eval_samples_b = ds_formatter["test"].select(range(min(200, len(ds_formatter["test"]))))

ass_tag = "<|start_header_id|>assistant<|end_header_id|>\n\n"

prompts_b = []
for sample in eval_samples_b:
    text = sample["text"]
    p_end = text.find(ass_tag)
    if p_end == -1:
        continue
    prompts_b.append(text[:p_end + len(ass_tag)])

print(f"Evaluating {len(prompts_b)} Formatter examples...")

json_ok = action_ok = 0
model_b.eval()

for batch_start in range(0, len(prompts_b), BATCH_SIZE):
    batch = prompts_b[batch_start : batch_start + BATCH_SIZE]

    tokenizer_b.padding_side = "left"
    enc = tokenizer_b(
        batch, return_tensors="pt", padding=True,
        truncation=True, max_length=512,
    ).to(DEVICE)

    with torch.no_grad():
        out_ids = model_b.generate(
            **enc, max_new_tokens=MAX_NEW_TOK,
            do_sample=False, pad_token_id=tokenizer_b.eos_token_id,
            eos_token_id=tokenizer_b.eos_token_id,
        )

    new_ids = out_ids[:, enc["input_ids"].shape[1]:]
    decoded = tokenizer_b.batch_decode(new_ids, skip_special_tokens=True)

    for gen in decoded:
        gen = gen.split("<|eot_id|>")[0].strip()
        try:
            parsed = json.loads(gen)
            json_ok += 1
            action_id = parsed.get("selected_action", {}).get("action_id", "")
            if action_id in VALID_ACTIONS:
                action_ok += 1
        except Exception:
            pass

    done = min(batch_start + BATCH_SIZE, len(prompts_b))
    print(f"  [{done}/{len(prompts_b)}] json_ok={json_ok}  action_ok={action_ok}")

tokenizer_b.padding_side = "right"
n = len(prompts_b)
print(f"\n{'='*50}")
print(f"Formatter Eval -- {n} examples:")
print(f"  JSON parse success : {json_ok/n*100:.1f}%  (target >=99%)")
print(f"  Valid action_id    : {action_ok/n*100:.1f}%  (target >=98%)")
print(f"{'='*50}")

---
# End-to-End Pipeline Test
Feed a test NPC state through both models sequentially:

**NPC state JSON** → **Reasoner (3B)** → **Turkish rationale** → **Formatter (1B)** → **JSON output**

In [ ]:
import json
import torch
from peft import PeftModel
from transformers import AutoTokenizer, AutoModelForCausalLM

# --- Reload Reasoner with adapter ---
tokenizer_a = AutoTokenizer.from_pretrained(ADAPTER_DIR_A)
tokenizer_a.pad_token = tokenizer_a.eos_token

base_a = AutoModelForCausalLM.from_pretrained(
    REASONER_MODEL_ID, torch_dtype=torch.float16, device_map="auto"
)
model_a = PeftModel.from_pretrained(base_a, ADAPTER_DIR_A)
model_a.eval()

# --- Constants ---
SYSTEM_PROMPT_REASONER = (
    "Sen bir ortacag simulasyonundaki NPC'nin ic ses motorusun.\n"
    "Sana NPC'nin anlik durumunu JSON olarak gonderecegim.\n"
    "Gorevin: NPC'nin ne yapmasi gerektigini ve NEDEN yapmasi gerektigini\n"
    "3-5 cumlelik Turkce ic monolog olarak yaz.\n"
    "ASLA JSON uretme. ASLA action_id adi kullanma. Sadece dusun.\n"
    "Kisilik ozellikleri, hayatta kalma ihtiyaclari, tehdit algisi,\n"
    "duygusal durum, hafiza ve sosyal baglami degerlendir."
)

SYSTEM_PROMPT_FORMATTER = (
    "Sen bir NPC simulasyonu icin JSON donusturucusun.\n"
    "Sana bir NPC'nin ne yapmak istedigini Turkce olarak anlatacagim.\n"
    "Bunu asagidaki JSON semasina donustur:\n"
    '{"npc_id":"<string>","reasoning":"<girdiyi kopyala>",'
    '"selected_action":{"action_id":"<listeden>","target_id":null,"dialogue":null},'
    '"emotion":"<tek kelime>"}\n'
    'valid_actions: ["eat","drink","sleep","flee","gather","heal",'
    '"attack","socialize","trade","work","pray","walk_to"]\n'
    "SADECE JSON yaz. Kod blogu veya aciklama ekleme."
)

TEMPLATE = (
    "<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\n"
    "{system}<|eot_id|><|start_header_id|>user<|end_header_id|>\n\n"
    "{user}<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n"
)

# --- Test state ---
test_state = {
    "id": "npc_demo01",
    "arch": "Brave", "occ": "Guard", "faction": "CityWatch",
    "b5": {"e":0.7,"a":0.5,"c":0.8,"n":0.2,"o":0.4},
    "traits": ["Brave","Loyal"],
    "vitals": {"hp":85.0,"hp_max":120.0,"en":0.65,"hun":0.80,"thi":0.45,"str":0.30},
    "emo": {"hap":0.3,"fear":0.1,"ang":0.2,"mood":"Calm"},
    "inv": [{"id":"food","n":1},{"id":"water","n":2}],
    "time": {"day":3,"hr":12.0},
    "pos": {"x":50.0,"z":50.0,"zone":"MarketSquare","landmark":"CentralFountain"},
    "sched": {"act":"work","wk_start":6,"wk_end":14,"sleep":21,"wake":5},
    "percepts": [{"id":"wolf_01","tag":"Threat","sal":0.92,"threat":0.85}],
    "memories": [{"evt":"Combat","desc":"Bir kurdu kovdum","ew":0.5,"dt":300}],
    "beliefs": [{"subj":"wolf_01","conf":0.8,"val":-0.9}],
    "factions": {"CityWatch":0.3,"Bandits":-0.7},
    "goals_top": "FindFood",
    "interrupt": True,
    "valid_actions": ["eat","drink","sleep","flee","gather","heal",
                      "attack","socialize","trade","work","pray","walk_to"]
}

state_json = json.dumps(test_state, ensure_ascii=False, separators=(',',':'))

# ==========================================
# STEP 1: Reasoner (3B) -> Turkish rationale
# ==========================================
prompt_a = TEMPLATE.format(system=SYSTEM_PROMPT_REASONER, user=state_json)

enc_a = tokenizer_a(prompt_a, return_tensors="pt", truncation=True, max_length=1024).to("cuda")
with torch.no_grad():
    out_a = model_a.generate(**enc_a, max_new_tokens=300, do_sample=False,
                             pad_token_id=tokenizer_a.eos_token_id)
new_a = out_a[:, enc_a["input_ids"].shape[1]:]
rationale = tokenizer_a.decode(new_a[0], skip_special_tokens=True).split("<|eot_id|>")[0].strip()

print("=" * 60)
print("STEP 1 -- Reasoner Output (Turkish Rationale):")
print("=" * 60)
print(rationale)
print()

# ==========================================
# STEP 2: Formatter (1B) -> JSON output
# ==========================================
prompt_b = TEMPLATE.format(system=SYSTEM_PROMPT_FORMATTER, user=rationale)

enc_b = tokenizer_b(prompt_b, return_tensors="pt", truncation=True, max_length=512).to("cuda")
with torch.no_grad():
    out_b = model_b.generate(**enc_b, max_new_tokens=200, do_sample=False,
                             pad_token_id=tokenizer_b.eos_token_id)
new_b = out_b[:, enc_b["input_ids"].shape[1]:]
json_output = tokenizer_b.decode(new_b[0], skip_special_tokens=True).split("<|eot_id|>")[0].strip()

print("=" * 60)
print("STEP 2 -- Formatter Output (JSON):")
print("=" * 60)
try:
    parsed = json.loads(json_output)
    print(json.dumps(parsed, indent=2, ensure_ascii=False))
    action = parsed.get("selected_action", {}).get("action_id", "MISSING")
    print(f"\n  action_id: {action}")
    print(f"  emotion:   {parsed.get('emotion', 'MISSING')}")
    print(f"  JSON valid: YES")
except json.JSONDecodeError as e:
    print(f"JSON PARSE ERROR: {e}")
    print(f"Raw output: {json_output}")

---
# Summary

| Component | Base | LoRA r | Train Examples | Epochs | Output Dir |
|-----------|------|--------|----------------|--------|------------|
| **A Reasoner** | Llama 3.2 3B | r=16 | 10k | 1 | `reasoner-lora-final/` |
| **B Formatter** | Llama 3.2 1B | r=8 | ~12k | 2 | `formatter-lora-final/` |

### Next Steps
1. Download both adapter folders from Kaggle output
2. Merge adapters into base models locally (`model.py` pattern)
3. Convert to GGUF Q4_K_M for Ollama deployment
4. Configure `DualLLMBackend` with two Ollama processes (ports 11434 / 11435)
5. Run simulation with the dual pipeline